# GridNet-HD — Heads lab: เทรน fusion head หลายแบบพร้อมกันบน feature ที่คำนวณไว้แล้ว (feature bank)

ทำไม: backbone ทั้งสองเป็น frozen → feature ของ LiDAR คำนวณ **ครั้งเดียว** ต่อโซนก็พอ เก็บเป็น "bank" (subset แบบ stratified ชุดเดียวกับที่ robustness protocol ใช้)
แล้ว 1 iteration ของ head เหลือแค่ gather จาก 2D maps + forward/backward ของ head ตัวเล็ก → เทรน head หลายแบบ **พร้อมกัน** บน batch ชุดเดียวกันได้
(วัดจริงบน A100, lab1: 12 heads ใน 3 batch groups = 0.55 s/it → 2 passes x 8 zones x 300 its ใน 54 นาที)

ขั้นตอน: (1) เตรียมข้อมูล (2) bank ของแต่ละโซน: มีบน Drive แล้วก็ copy มาใช้ ไม่มีก็สร้าง (lab1: 26 นาทีสำหรับ 14 โซน ตอนนี้อยู่บน Drive ครบแล้ว)
(3) เทรนทุก head ใน `SPECS` พร้อมกัน validate บน bank ของ val zones ทุก pass (4) diagnostics: น้ำหนักที่ดีที่สุดของทุก head ถูกวัดแบบ
**ไม่มี LiDAR** / **ไม่มีภาพ** และบน **train zones** (โซนที่ backbone ทั้งสองเคยเห็นตอนเทรน) — head เดิมใน `REFERENCE_HEADS`
(เช่น `late_v1` ที่เทรนแบบ crop) ถูกวัดบน bank ชุดเดียวกันไปด้วยโดยไม่เทรนใหม่ (5) `PROTOCOL`: robustness protocol (โหมด fast) ของทุก head ใน lab นี้
+ reference heads → `GridNet/protocol/<TAG>/` (6) `FULL_EVAL`: full-zone evaluation ของ head ที่ระบุ (~20 นาที/head)

**lab4 = รอบสุดท้ายของการทดลอง** (หลัง lab3: G5 ครบ 3 seeds, fix = pose-noise augmentation): เทรน 4 heads ที่ยังขาด
(seeds 1 และ 2 ของ window K=5 แบบไม่มีและมี augmentation — `win5_n` seed 0 เสีย thin classes น้อยกว่า concat / K=3 ราวครึ่งหนึ่ง จึงต้องมี error bar)
แล้วรัน **protocol โหมด exact กับโมเดลทุกตัวในตารางของ paper** (`PROTOCOL_HEADS`: 35 heads + LiDAR-only + ImageVote → `GridNet/protocol/exact_v1/`)
และ **ablation จำนวน views ตอน test** (cell 8b: โหมด fast กับ V = 1, 2, 4, 8 views ที่ใกล้ที่สุด → `GridNet/protocol/views<V>/`)
เวลาบน A100: วัดจริงจาก lab3 (12 heads ใน 8 groups, 13 พ.ย.) = เตรียมข้อมูล 26 นาที, เทรน 58 นาที (0.58 s/it) + diagnostics 8 นาที, protocol โหมด fast 37 นาทีกับ 18 โมเดล
(3.4 s ต่อ configuration), full-zone evaluation 18 นาที; lab4 ประมาณจากอัตราเหล่านี้ (ยังไม่ได้วัด): เทรน ~50 นาที, protocol โหมด exact ~2.5–3 ชม.
(654 configurations × ราว 11–16 s), views ~35 นาที → ทั้ง notebook ราว 4.5–5 ชม. · resume ได้ทุกขั้น (protocol ต่อจากโซนที่ค้าง)

`SPECS` (cell 2): หนึ่งบรรทัดต่อ head รูปแบบ `name:head[:key=value,...]` · head = `late` | `concat` | `window`
keys: `k`, `lidar_input` (`feat` | `prob`), `image_input` (`feat` | `prob`, concat เท่านั้น), `lidar_drop`, `natural_frac`,
`pose_t`, `pose_r`, `aug_prob`, `seed`, `lr`, `wd`
ผลของแต่ละ head: `GridNet/fusion/<name>/best.pth` (ใช้กับ `colab_protocol` ได้เลย) + `eval_bank/eval.json` + `eval_bank/diagnostics.json`; ตารางรวม: `GridNet/fusion/bank_<TAG>.json`

**Resume**: Run all ซ้ำได้ — head ที่เทรนจบแล้วถูกข้าม ถ้าเพิ่มบรรทัดใหม่ใน `SPECS` จะเทรนเฉพาะตัวใหม่ (ชื่อ head ห้ามซ้ำกับ run เดิมที่ตั้งค่าต่างกัน)

In [ ]:
#@title 1. GPU (this notebook needs the 40 GB of an A100: the 2D maps of t6z2 / t6z4 alone take ~14 GB on the GPU)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.get_device_capability()[0] >= 8, "need an A100 (flash-attn)"
mem = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split()[0])
assert mem >= 30000, ("Colab connected a smaller GPU than the A100 this notebook asks for (the A100 was not free, or another of your sessions still holds it). "
                      "Runtime -> Manage sessions: terminate old sessions; Runtime -> Disconnect and delete runtime; then run this cell again")


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
TAG = 'lab4'                                     #@param {type:"string"}
# one head per line: name:head[:key=value,...]   (heads with the same seed and sampling share their batches)
# _n = batches at the natural class frequencies (default); _a = pose-noise augmentation: on half of the iterations
# every camera gets i.i.d. noise with sigma_t ~ U(0, pose_t) m and sigma_r ~ U(0, pose_r) deg; _sK = seed K
# lab4: seeds 1 and 2 of the 5x5 window, without and with augmentation (seed 0 = win5_n of lab3, win5_a of lab2)
SPECS = '''
win5_n_s1:window:k=5,seed=1
win5_n_s2:window:k=5,seed=2
win5_a_s1:window:k=5,pose_t=0.5,pose_r=1.0,seed=1
win5_a_s2:window:k=5,pose_t=0.5,pose_r=1.0,seed=2
'''
REFERENCE_HEADS = 'win5_n,win5_a'                #@param {type:"string"}
PROTOCOL = True                                  #@param {type:"boolean"}
PROTOCOL_MODE = 'exact'                          #@param ["fast", "exact"]
PROTOCOL_DRAWS = 10                              #@param {type:"integer"}
PROTOCOL_TAG = 'exact_v1'                        #@param {type:"string"}
# the heads of the protocol run, in the order of the paper tables ('' = the reference heads + the heads of SPECS)
PROTOCOL_HEADS = '''
late_nat_b1 late_n_s1 late_n_s2
concat_n concat_n_s1 concat_n_s2
win3_n win3_n_s1 win3_n_s2
win5_n win5_n_s1 win5_n_s2
late_a late_a_s1 late_a_s2
concat_a concat_a_s1 concat_a_s2
win3_a win3_a_s1 win3_a_s2
win5_a win5_a_s1 win5_a_s2
late_a10 late_a25 late_a100
concat_a10 concat_a25 concat_a100
concat_at concat_ar
concat_ip_n concat_lp_n concat_lp_a
'''
PROTOCOL_PLOT = 'lidar,imagevote,late_nat_b1,late_a,concat_n,concat_a,win3_n,win5_n'  #@param {type:"string"}
# number of views at test time (cell 8b): fast protocol with the V nearest views, '' = off
PROTOCOL_VIEWS = '1,2,4,8'                       #@param {type:"string"}
VIEWS_HEADS = 'late_nat_b1,late_a,concat_n,concat_a'  #@param {type:"string"}
VIEWS_DRAWS = 5                                  #@param {type:"integer"}
FULL_EVAL = ''                                   #@param {type:"string"}
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
BACKBONE_CKPT = f'{DRIVE}/ckpt/litept_s_lidar_v2/best.pth'   #@param {type:"string"}
TRAIN_ZONES = 't1z6a,t2z5,t3z3,t5a1,t5b2,t5c1,t6z2,t6z4'      #@param {type:"string"}
VAL_ZONES = 't1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'                  #@param {type:"string"}
PASSES = 2                #@param {type:"integer"}
ITS_PER_ZONE = 300        #@param {type:"integer"}
BATCH_POINTS = 120000     #@param {type:"integer"}
DIAGNOSTICS = True        #@param {type:"boolean"}
MAX_HOURS = 5.0           #@param {type:"number"}
SHUTDOWN_WHEN_DONE = False #@param {type:"boolean"}
CACHE_DRIVE = f'{DRIVE}/cache'
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'
ASSOC_DIR = f'{DRIVE}/fusion/assoc_{RUN2D}'
BANK_ARCHIVE = f'{DRIVE}/protocol/subsets_{RUN2D}'   # <zone>/subset.npz: one file per zone, shared with colab_protocol
OUT_ROOT = f'{DRIVE}/fusion'
for d in (ASSOC_DIR, BANK_ARCHIVE, OUT_ROOT): os.makedirs(d, exist_ok=True)
train_zones = [z.strip() for z in TRAIN_ZONES.split(',') if z.strip()]
val_zones = [z.strip() for z in VAL_ZONES.split(',') if z.strip()]
specs = SPECS.replace(';', ' ').split()
for sp in specs:
    f = sp.split(':')
    assert len(f) in (2, 3) and f[1] in ('late', 'concat', 'window'), f"bad spec {sp!r}: use name:head[:key=value,...]"
names = [sp.split(':')[0] for sp in specs]
assert len(set(names)) == len(names), "every head needs its own name"
references = [n.strip() for n in REFERENCE_HEADS.split(',') if n.strip()]
assert not set(references) & set(names), "a head cannot be trained and be a reference at the same time"
full_eval = [n.strip() for n in FULL_EVAL.split(',') if n.strip()]
protocol_heads = PROTOCOL_HEADS.replace(',', ' ').split() or references + names
assert len(set(protocol_heads)) == len(protocol_heads), "a head is listed twice in PROTOCOL_HEADS"
views = [int(v) for v in PROTOCOL_VIEWS.replace(',', ' ').split()]
views_heads = [n.strip() for n in VIEWS_HEADS.split(',') if n.strip()]
have_bank = {z: os.path.exists(f'{BANK_ARCHIVE}/{z}/subset.npz') for z in train_zones + val_zones}
pose_noise = any('pose_t=' in sp or 'pose_r=' in sp for sp in specs)
print(len(specs), "heads:", ' '.join(names))
print("reference heads (evaluated on the banks, not trained):", [n for n in references if os.path.exists(f'{OUT_ROOT}/{n}/best.pth')] or 'none',
      "| missing:", [n for n in references if not os.path.exists(f'{OUT_ROOT}/{n}/best.pth')] or 'none')
print("already trained (best.pth on Drive):", [n for n in names if os.path.exists(f'{OUT_ROOT}/{n}/best.pth')] or 'none')
print(f"banks on Drive: {sum(have_bank.values())}/{len(have_bank)}", "" if all(have_bank.values()) else "-> the missing ones are built in cell 6 (~3-8 min per zone)")
PTAG = PROTOCOL_TAG.strip() or TAG
print("protocol after training:", f"{PROTOCOL_MODE} mode, {len(protocol_heads)} heads -> {DRIVE}/protocol/{PTAG}" if PROTOCOL else "off")
_missing = [n for n in protocol_heads if n not in names and not os.path.exists(f'{OUT_ROOT}/{n}/best.pth')]
if PROTOCOL and _missing: print("   NOT on Drive and not trained here (they would be left out of the protocol):", _missing)
print("views at test time (cell 8b):", f"V = {views} with {views_heads}, fast mode, {VIEWS_DRAWS} draws" if views else "off")


In [ ]:
#@title 3. Environment (restore tarball from Drive)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    assert os.path.exists(TAR), "no env tarball on Drive: run colab_train_litept cell 3 once"
    subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pillow', 'matplotlib'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code + LitePT
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
assert 'def nearest_views' in open('/content/code/gridnet/train/protocol.py').read(), "gridnet_code.zip on Drive is older than update34: pack and upload it again"
if not os.path.exists('/content/LitePT'):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/prs-eth/LitePT.git", "/content/LitePT"], check=True)
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/LitePT'); sys.path.insert(0,'/content/code'); "
                    "from litept.model import LitePT; import gridnet.train.bank; print('imports ok')"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])


In [ ]:
#@title 5. Data -> local disk: 2D caches (zone tars), camera files; LiDAR caches where needed (bank to build, pose noise, protocol, FULL_EVAL)
import os, sys, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '/content/code')
from gridnet.train.maps2d import cached_image_ids, zone_tar_manifest, extract_zone_tar, write_compact
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
zones = train_zones + val_zones
need_points = [z for z in zones if not have_bank[z] or (pose_noise and z in train_zones) or ((full_eval or PROTOCOL) and z in val_zones)]
t0 = time.time()
for z in need_points:
    dst = f'/content/cache/{z}/points.npz'
    if not os.path.exists(dst):
        os.makedirs(f'/content/cache/{z}', exist_ok=True); shutil.copy(f'{CACHE_DRIVE}/{z}/points.npz', dst)
print(f"LiDAR caches: {len(need_points)} zones ({(time.time() - t0) / 60:.1f} min)" + ("" if need_points else " - none needed"))
CACHE2D = '/content/cache2d'; t0 = time.time(); slow = []
for z in zones:
    if zone_tar_manifest(CACHE2D_DRIVE, z, wait=30) is not None:   # one big sequential read: fast
        n = extract_zone_tar(CACHE2D_DRIVE, z, CACHE2D); how = 'tar'
    else:                                                             # no tar yet: parallel per-file copy (slow)
        ids = cached_image_ids(CACHE2D_DRIVE, z)
        with ThreadPoolExecutor(16) as ex:
            list(ex.map(lambda i: write_compact(CACHE2D_DRIVE, z, i, CACHE2D), ids))
        n = len(cached_image_ids(CACHE2D, z)); how = 'files'; slow.append(z)
    print(f"  {z}: {n} images from {how}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
if slow: print("zones without a tar (slow path) - run colab_compact_cache2d once for:", ','.join(slow))
code = ("import sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
        f"fetch_zones({zones!r}, '{DATA}', what=('pose',))")
subprocess.run([PY, '-c', code], check=True); print("camera files ready")
CKPT_LOCAL = '/content/backbone_best.pth'
if (full_eval or not all(have_bank.values())) and not os.path.exists(CKPT_LOCAL): shutil.copy(BACKBONE_CKPT, CKPT_LOCAL)   # the protocol reads the banks: no backbone
print(subprocess.run(['df', '-h', '--output=avail', '/content'], capture_output=True, text=True).stdout.split()[-1], "free on the local disk")


In [ ]:
#@title 6. Banks (copy from Drive, or build + archive) -> train every head of SPECS -> diagnostics
import subprocess, time, os
cmd = [PY, '/content/code/scripts/train_heads_bank.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
       '--assoc-dir', ASSOC_DIR, '--subset-dir', '/content/banks', '--subset-archive', BANK_ARCHIVE, '--litept', '/content/LitePT',
       '--out-root', OUT_ROOT, '--specs', ' '.join(specs), '--tag', TAG, '--train-zones', ','.join(train_zones),
       '--val-zones', ','.join(val_zones), '--passes', str(PASSES), '--its-per-zone', str(ITS_PER_ZONE),
       '--batch-points', str(BATCH_POINTS), '--max-hours', str(MAX_HOURS), '--references', ','.join(references)]
if os.path.exists(CKPT_LOCAL): cmd += ['--backbone-ckpt', CKPT_LOCAL]
if DIAGNOSTICS: cmd += ['--diagnostics']
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        if 'Warning' not in line and 'def backward' not in line and '_TORCH_CUSTOM_FWD' not in line: print(line, end='')
    p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
except KeyboardInterrupt:
    p.terminate(); p.wait(); print("stopped by user; finished passes and banks are kept -> re-run this cell to continue"); raise


In [ ]:
#@title 7. Tables of this group (GridNet/fusion/bank_<TAG>.json)
import json, os
path = f'{OUT_ROOT}/bank_{TAG}.json'
def show(title, metrics, order):
    rows = [n for n in order if n in metrics]
    if not rows: return
    classes = list(metrics[rows[0]]['per_class'])
    print(f"=== {title} ===")
    print(f"{'model':20s} {'mIoU':>6s} {'OA':>6s}  " + ' '.join(f"{c[:9]:>9s}" for c in classes))
    for n in rows:
        m = metrics[n]
        print(f"{n:20s} {m['mIoU']:6.2f} {m['OA']:6.2f}  " + ' '.join(f"{'--' if m['per_class'][c] is None else format(m['per_class'][c], '.1f'):>9s}" for c in classes))
    print()
if os.path.exists(path):
    s = json.load(open(path)); order = ['lidar', 'imagevote'] + s.get('references', []) + [sp['name'] for sp in s['specs']]
    titles = {'val': 'validation zones, best weights (bank estimate)', 'val_no_lidar': 'validation, heads WITHOUT LiDAR evidence',
              'val_no_image': 'validation, heads WITHOUT image evidence', 'train': 'training zones (both backbones were trained on them)'}
    for case, t in titles.items():
        if case in s['diagnostics']: show(t, s['diagnostics'][case], order)
    val = s['diagnostics']['val']; rows = [n for n in order if n in val]
    zs = list(val[rows[0]]['per_zone'])
    print("=== validation mIoU per zone (classes present in the zone) ===")
    print(f"{'model':20s} " + ' '.join(f"{z:>7s}" for z in zs))
    for n in rows: print(f"{n:20s} " + ' '.join(f"{val[n]['per_zone'][z]:7.2f}" for z in zs))
else:
    print("no summary yet:", path, "(cell 6 writes it once every head has finished)")


In [ ]:
#@title 8. Robustness protocol of the heads in PROTOCOL_HEADS -> GridNet/protocol/<PROTOCOL_TAG>/ (654 configurations: fast ~0.4 s + 0.17 s per model each; exact ~5-10 s more each; one progress line per noise level)
import subprocess, time, os
from IPython.display import Image, display
if PROTOCOL:
    POUT = f'{DRIVE}/protocol/{PTAG}'
    pheads = {n: f'{OUT_ROOT}/{n}/best.pth' for n in protocol_heads if os.path.exists(f'{OUT_ROOT}/{n}/best.pth')}
    left_out = [n for n in protocol_heads if n not in pheads]
    if left_out: print("WARNING - no best.pth, LEFT OUT of the protocol:", left_out)
    print(f"{len(pheads)} heads + LiDAR-only + ImageVote, {PROTOCOL_MODE} mode")
    local = all(os.path.exists(f'/content/banks/{z}/subset.npz') for z in val_zones)
    cmd = [PY, '/content/code/scripts/run_protocol.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
           '--assoc-dir', ASSOC_DIR, '--out', POUT, '--subset-dir', '/content/banks' if local else BANK_ARCHIVE, '--litept', '/content/LitePT',
           '--zones', ','.join(val_zones), '--sweeps', 'trans,rot,drift', '--draws', str(PROTOCOL_DRAWS), '--mode', PROTOCOL_MODE,
           '--heads'] + [f'{n}={p}' for n, p in pheads.items()]
    env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True', MPLBACKEND='Agg')
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    last = None
    try:
        for line in p.stdout:                               # one line per noise level and per zone (per-draw numbers: results.json)
            if line.startswith('[protocol]') and ' draw ' in line:
                key = ' '.join(line.split()[1:4])           # zone sweep level
                if key != last: print(f"   {key}  ({(time.time() - t0) / 60:.1f} min)", flush=True); last = key
            elif line.startswith('[protocol]'): print(line, end='')
            elif 'Error' in line or 'Traceback' in line or line.startswith('  File'): print(line, end='')
    except KeyboardInterrupt:
        p.terminate(); p.wait(); print("stopped by user; finished zones are in results.json -> run this cell again (Runtime -> Run after) to continue"); raise
    p.wait(); print(f"protocol: exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
    if p.returncode != 0: print("-> if it STOPPED on 'other settings' (heads were added since the first run): give PROTOCOL_TAG a new name in cell 2, re-run cell 2 and this cell")
    fig = f'{POUT}/protocol_{PTAG}.png'
    r = subprocess.run([PY, '/content/code/scripts/plot_protocol.py', '--summary', f'{POUT}/summary.json', '--out', fig.replace('.png', '_all.png'),
                        '--tables', 'compact'], capture_output=True, text=True, env=env)        # table: every model
    print(r.stdout[-9000:], r.stderr[-800:])
    r = subprocess.run([PY, '/content/code/scripts/plot_protocol.py', '--summary', f'{POUT}/summary.json', '--out', fig, '--tables', 'compact',
                        '--models', PROTOCOL_PLOT], capture_output=True, text=True, env=env)     # the figure: a readable selection
    display(Image(fig)) if os.path.exists(fig) else print(r.stderr[-800:])
else:
    print("PROTOCOL is off")


In [ ]:
#@title 8b. (optional) Number of views at test time: fast protocol with the V nearest views (PROTOCOL_VIEWS) -> GridNet/protocol/views<V>/ (ImageVote + VIEWS_HEADS; ~1.3 s per configuration, ~8 min per V with 5 draws)
import subprocess, time, os, json
if views:
    vheads = {n: f'{OUT_ROOT}/{n}/best.pth' for n in views_heads if os.path.exists(f'{OUT_ROOT}/{n}/best.pth')}
    local = all(os.path.exists(f'/content/banks/{z}/subset.npz') for z in val_zones)
    env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True', MPLBACKEND='Agg')
    for V in views:
        cmd = [PY, '/content/code/scripts/run_protocol.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
               '--assoc-dir', ASSOC_DIR, '--out', f'{DRIVE}/protocol/views{V}', '--subset-dir', '/content/banks' if local else BANK_ARCHIVE,
               '--litept', '/content/LitePT', '--zones', ','.join(val_zones), '--sweeps', 'trans,rot,drift', '--draws', str(VIEWS_DRAWS),
               '--mode', 'fast', '--views', str(V), '--no-lidar', '--heads'] + [f'{n}={p}' for n, p in vheads.items()]
        t0 = time.time()
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        try:
            for line in p.stdout:                               # one line per zone
                if line.startswith('[protocol]') and ' draw ' not in line and ' head ' not in line: print(f"   V={V}  {line}", end='', flush=True)
                elif 'Error' in line or 'Traceback' in line or line.startswith('  File'): print(line, end='')
        except KeyboardInterrupt:
            p.terminate(); p.wait(); print("stopped by user; finished zones are in results.json -> run this cell again to continue"); raise
        p.wait(); print(f"views {V}: exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
        if p.returncode != 0: print(f"-> if it STOPPED on 'other settings': GridNet/protocol/views{V} was started with other heads or draws; rename that folder on Drive and run this cell again")
    done = {}
    for V in views:
        f = f'{DRIVE}/protocol/views{V}/summary.json'
        if os.path.exists(f):
            sm = json.load(open(f))
            done[V] = (sm['meta']['models'], {(c['model'], c['sweep'], float(c['level']), c['class']): 100 * c['iou_mean'] for c in sm['curves'] if c['iou_mean'] is not None})
    cols = [('trans', 0.0, 'clean'), ('trans', 0.1, 'trans 0.1'), ('trans', 0.5, 'trans 0.5'), ('rot', 0.1, 'rot 0.1'), ('rot', 1.0, 'rot 1'), ('drift', 0.5, 'drift 0.5')]
    if done:
        print()
        print(f"mean IoU [%]: all classes / electrical classes, with the V nearest views (fast mode, {VIEWS_DRAWS} draws)")
        print(f"{'model':14s}{'V':>3s}" + ''.join(f"{c[2]:>15s}" for c in cols))
        for m in done[min(done)][0]:
            for V in sorted(done):
                val = done[V][1]
                print(f"{m:14s}{V:3d}" + ''.join(f"{val[(m, sw, lv, 'mean_all')]:.1f} / {val[(m, sw, lv, 'mean_electrical')]:.1f}".rjust(15) for sw, lv, _ in cols))
else:
    print("PROTOCOL_VIEWS is empty")


In [ ]:
#@title 9. (optional) Full-zone evaluation of the heads in FULL_EVAL: backbone on every window + window votes (~20 min per head, a few lines per zone while it runs) -> <head>/eval/eval.json
import subprocess, json, os, time
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
for name in full_eval:
    work = f'{OUT_ROOT}/{name}'; best = f'{work}/best.pth'; ev = f'{work}/eval/eval.json'
    if not os.path.exists(best):
        print(f"[{name}] no best.pth - skipped"); continue
    if not (os.path.exists(ev) and os.path.getmtime(ev) >= os.path.getmtime(best)):      # missing, or older than the weights
        cmd = [PY, '/content/code/scripts/train_fusion.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
               '--assoc-dir', ASSOC_DIR, '--litept', '/content/LitePT', '--backbone-ckpt', CKPT_LOCAL, '--work-dir', work,
               '--val-zones', ','.join(val_zones), '--eval-only']
        t0 = time.time(); tail = []
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        try:
            for line in p.stdout:                                        # the zone being loaded / evaluated, with the minutes so far
                if 'Warning' in line: continue
                tail = (tail + [line])[-40:]
                if line.startswith('[eval]') or line.startswith('[fusion]'): print(f"   {(time.time() - t0) / 60:5.1f} min  {line}", end='', flush=True)
        except KeyboardInterrupt:
            p.terminate(); p.wait(); print("stopped by user -> run this cell again to evaluate this head from the start"); raise
        p.wait()
        if p.returncode != 0: print(f"[{name}] full-zone evaluation FAILED (exit {p.returncode}); last lines:"); print(''.join(tail)[-2500:])
    if not os.path.exists(ev): continue
    e = json.load(open(ev)); b = f'{work}/eval_bank/eval.json'
    print(f"[{name}] full-zone evaluation: fused mIoU {e['mIoU']:.2f}  OA {e['OA']:.2f}   LiDAR-only {e['lidar_only']['mIoU']:.2f}"
          + (f"   | bank estimate {json.load(open(b))['mIoU']:.2f}" if os.path.exists(b) else ''))
    print('  fused     ', {k: (None if v is None else round(v, 1)) for k, v in e['per_class'].items()})
    print('  per zone  ', {z: round(v['mIoU'], 1) for z, v in e['per_zone'].items()})


In [ ]:
#@title 10. (optional) Shut the runtime down once everything above has finished - saves compute units on unattended runs
if SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    print("flushing pending writes to Drive ..."); drive.flush_and_unmount()
    print("results are on Drive; shutting the runtime down"); runtime.unassign()
else:
    print("SHUTDOWN_WHEN_DONE is off -> remember to Disconnect and delete runtime")


## หมายเหตุ
- ตัวเลขในตารางเป็น **bank estimate**: วัดบน subset แบบ stratified แล้วคูณ weight กลับเป็นทั้งโซน (= level 0 ของ robustness protocol)
  โดยใช้ LiDAR feature ที่เฉลี่ยข้าม window แล้ว ส่วน full-zone evaluation (cell 9, `colab_train_fusion`) ใช้ head กับทุก window แล้วเฉลี่ย vote
- ตรวจก่อนเชื่อตารางอื่น: แถว `lidar` ต้องได้ใกล้ 69.9 และ `imagevote` ใกล้ 65.5 (ต่างเกิน ~0.5 = มีอะไรผิด ให้หยุด);
  `late_v1` บน bank เทียบกับ 82.0 จาก full evaluation บอกว่าสองวิธีวัดต่างกันเท่าไร
- lab1 (13 พ.ย.): head ที่เทรนจาก bank ด้วย batch ตามความถี่จริงของคลาส (`late_nat_b1` 81.8) ได้เท่ากับ head ที่เทรนแบบ crop (`late_v1` 81.9)
  ส่วน `natural_frac=0.5` (head ชื่อ `*_b1` ที่เหลือ) เสียราว 1.4 mIoU → ตั้งแต่ update31 default คือ `natural_frac=1.0`
- ชื่อย่อใน lab1: `_lp` = head เห็น LiDAR เป็น class probabilities เท่านั้น (`lidar_input=prob`), `_ip` = เห็นภาพเป็น probabilities เท่านั้น
  (`image_input=prob`), `_ld50` = LiDAR dropout 50 % ตอนเทรน, `_s1` = seed 1; ใน lab2: `_n` = batch ตามความถี่จริง
- head ที่ `seed` และ `natural_frac` เดียวกันเห็น batch ชุดเดียวกัน (เทียบแบบ paired); `seed=1` ได้ batch + initialisation ชุดใหม่
- `natural_frac` = สัดส่วนของ batch ที่ sample ตามความถี่จริงของคลาส (ที่เหลือ sample แบบ uniform จาก bank = rare classes ถูกเน้น); default 1.0
- cell 6 พิมพ์ตารางจำนวนจุดต่อคลาสของทุกโซน (หน่วยพันจุด) ก่อนเทรน: ใช้ดูว่าคลาสหายากอย่าง water อยู่ในโซนไหนบ้าง
- pose-noise augmentation (`pose_t`, `pose_r` ใน spec): ในครึ่งหนึ่งของ iterations (`aug_prob`) กล้องทุกตัวได้ noise แบบ i.i.d. ที่ σ_t ~ U(0, `pose_t`) m และ
  σ_r ~ U(0, `pose_r`) ° แล้ว re-project แบบเดียวกับ protocol โหมด fast (cell 5 จะ copy LiDAR cache ของ train zones ให้เอง)
- cell 8 (`PROTOCOL`): ตารางหนึ่งแถวต่อโมเดล = mean IoU ทุก class / เฉลี่ย 4 electrical classes ที่ clean → reference level → level ใหญ่สุดของแต่ละ sweep;
  คอลัมน์ `clean` ต้องเท่ากับ mIoU ของตาราง validation ใน cell 7; รูปแสดงเฉพาะโมเดลใน `PROTOCOL_PLOT` (ทุกโมเดล: `protocol_<TAG>_all.png`);
  ถ้า Run all ซ้ำหลังเพิ่ม head ใน `SPECS` ให้ตั้ง `PROTOCOL_TAG` ใหม่ (results.json เดิมต่อได้เฉพาะชุดโมเดลเดิม)
- ผลของ gate G3 (run `fast_v1`): ที่ 10 cm / 0.1° ทุก head เสียไม่เกิน 0.3 mIoU; ที่ 50 cm / 1° head ระดับ feature (concat, window) เสีย thin classes
  7–11 จุด ส่วน late เสีย ≤ 1.4 → lab2 ถามว่า pose-noise augmentation ปิดช่องว่างนั้นได้ไหม และ window K=3 / K=5 ช่วยเพิ่มหรือไม่
- ผลของ lab2 (protocol `lab2`, 13 พ.ย., เฉลี่ยข้าม seeds; mIoU ทุก class / electrical classes ที่ clean → 50 cm → 1°):
  late 81.8/98.0 → 80.2/96.6 → 76.8/95.8 · concat 79.2/98.0 → 74.2/86.2 → 70.0/80.9 · window K=3 79.6/98.0 → 74.7/86.5 → 69.7/79.8 ·
  late+aug 82.1/98.0 → 81.3/97.8 → 78.8/97.8 · concat+aug 81.1/98.1 → 80.3/97.9 → 77.8/97.8 · window K=3+aug 80.7/98.1 → 79.9/97.9 → 77.5/97.9
  → augmentation คือ fix (clean ไม่ตก), window K=3 / K=5 ไม่ช่วยเพิ่ม
- cell 8 ตั้งแต่ update33: ตารางมีบรรทัด "median displacement of a projected point [px]" (จุดที่ project ลงภาพขยับกี่ pixel ที่แต่ละระดับ noise)
  และเขียน `confusion.json` + `results.json.gz` ไว้ข้าง `summary.json`
- ตรวจผลของ lab3: reference heads ใช้กล้องชุดเดียวกับ lab2 (draw ถูก seed ด้วยชื่อ sweep + ระดับ) จึงต้องได้ตัวเลขเดิม (clean | trans 0.5 | rot 1):
  `late_nat_b1` 81.8/98.0 | 80.7/97.5 | 77.6/97.1 · `late_a` 82.1/98.0 | 81.4/97.8 | 79.0/97.8 · `concat_n` 79.4/98.1 | 74.8/87.3 | 70.5/82.0 ·
  `concat_a` 81.7/98.1 | 80.8/97.9 | 78.2/97.8 (ต่างเกิน 0.1 = มีอะไรเปลี่ยน ให้บอกก่อนใช้ผล)
- ผลของ lab3 (13 พ.ย.): head ที่มี augmentation ครบ 3 seeds — late 82.05 ± 0.02, concat 80.65 ± 0.98, window K=3 80.57 ± 0.71 (clean) และ electrical 97.8–97.9 ที่ 50 cm / 1° ทุกตัว (G5: K×K ไม่ชนะ augmentation);
  `win5_n` (K=5 ไม่มี augmentation) เสีย thin classes น้อยกว่า concat / K=3 ราวครึ่งหนึ่ง (92.6 / 88.0 เทียบ 87.3 / 82.0) → lab4 เทรน seeds เพิ่ม; noise ≤10 cm ตอนเทรนก็ช่วยแล้ว (96.2 / 94.8),
  ชนิดของ noise ไม่สำคัญ, `concat_lp_a` = late + aug; full-zone evaluation ของ `late_a` = 82.06 (bank 82.07); ระยะขยับ median: 10 cm = 6.4 px, 50 cm = 31.9 px, 0.1° = 8.3 px, 1° = 82.8 px
- lab4: cell 8 รันโหมด **exact** (สร้าง association ใหม่ทุก draw) กับ 37 โมเดล — ช้ากว่าโหมด fast: บรรทัดความคืบหน้าขึ้นทุก ~2–3 นาที (หนึ่งระดับ noise = 10 draws);
  ถ้า session หลุด ให้ Run all ใหม่ได้เลย: heads ที่เทรนแล้วถูกข้าม และ protocol ต่อจากโซนที่ค้าง
- cell 8b: จำนวน views ตอน test (V views ที่ใกล้ที่สุดจาก 8 ที่เก็บไว้) กับ ImageVote + `VIEWS_HEADS` ในโหมด fast, 5 draws → ตารางเทียบ V ท้าย cell
- จบแล้ว **Disconnect and delete runtime**